# Zero-Shot Video Classification

## Task Description
Classify a video into arbitrary categories without specific training, using a Zero-Shot model.

**Focus:** This notebook uses `microsoft/xclip-base-patch32`, an implementation of CLIP adapted for video.

In this notebook, we will:
1. **Install:** Install `transformers` and `av` (for video processing).
2. **Load Data:** Download a sample video.
3. **Load Model:** Load X-CLIP.
4. **Inference:** Classify the video against custom text labels.
5. **Visualize:** Display results.

## License

This notebook is licensed under the Apache License 2.0. See the [LICENSE](https://github.com/huggingface/transformers/blob/main/LICENSE) file for more details.

This notebook focuses on open-source models and is optimized to run on the Google Colab Free Tier.

## 1. Installation
We need `av` for decoding video frames.

In [ ]:
!pip install -q --upgrade transformers torch av huggingface_hub

## 2. Imports


In [ ]:
import torch
import numpy as np
from transformers import XClipProcessor, XClipModel
from huggingface_hub import hf_hub_download
import av

## 3. Load Data
Function to read video frames.

In [ ]:
def read_video_pyav(container, indices):
    '''
    Decode the video with PyAV decoder.
    Args:
        container (av.container.input.InputContainer): PyAV container.
        indices (List[int]): List of frame indices to decode.
    Returns:
        result (np.ndarray): np array of decoded frames of shape (num_frames, height, width, 3).
    '''
    frames = []
    container.seek(0)
    start_index = indices[0]
    end_index = indices[-1]
    for i, frame in enumerate(container.decode(video=0)):
        if i > end_index:
            break
        if i >= start_index and i in indices:
            frames.append(frame)
    return np.stack([x.to_ndarray(format="rgb24") for x in frames])

def sample_frame_indices(clip_len, frame_sample_rate, seg_len):
    converted_len = int(clip_len * frame_sample_rate)
    end_idx = np.random.randint(converted_len, seg_len)
    start_idx = end_idx - converted_len
    indices = np.linspace(start_idx, end_idx, num=clip_len)
    indices = np.clip(indices, start_idx, end_idx - 1).astype(np.int64)
    return indices

# Download sample video
video_path = hf_hub_download(repo_id="microsoft/xclip-base-patch32", filename="eating_spaghetti.mp4", repo_type="model")
print(f"Video downloaded to: {video_path}")

## 4. Data Cleaning
Sample frames.

In [ ]:
container = av.open(video_path)

# Sampling logic
indices = sample_frame_indices(clip_len=8, frame_sample_rate=1, seg_len=container.streams.video[0].frames)
video = read_video_pyav(container, indices)
print(f"Video shape: {video.shape}")

## 5. Load Model
Load `microsoft/xclip-base-patch32`.

In [ ]:
model_name = "microsoft/xclip-base-patch32"
processor = XClipProcessor.from_pretrained(model_name)
model = XClipModel.from_pretrained(model_name)

print("X-CLIP loaded.")

## 6. Inference
Define candidate labels and run inference.

In [ ]:
text_labels = ["playing football", "eating spaghetti", "walking the dog"]

inputs = processor(text=text_labels, videos=list(video), return_tensors="pt", padding=True)

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)

probs = outputs.logits_per_video.softmax(dim=1)

print("Probabilities:")
for label, prob in zip(text_labels, probs[0]):
    print(f"{label}: {prob:.4f}")

## 7. Save Results


In [ ]:
best_label = text_labels[probs.argmax()]
with open('results.txt', 'w') as f:
    f.write(f"Best Label: {best_label}")
print('Results saved to results.txt')

# Kitchen Sink: Save Results
import os
import numpy as np
try:
    from PIL import Image
except ImportError:
    pass
try:
    import scipy.io.wavfile as wavfile
except ImportError:
    pass

print("Saving results...")

# 1. Text / Data / Variables
with open('results.txt', 'w') as f:
    if 'results' in locals():
        f.write(str(results))
    elif 'predictions' in locals():
        f.write(str(predictions))
    elif 'prediction' in locals():
        f.write(str(prediction))
    elif 'output' in locals():
        f.write(str(output))
    elif 'features' in locals():
        f.write(str(features))
    elif 'summary' in locals():
        f.write(str(summary))
    elif 'translation' in locals():
        f.write(str(translation))
    elif 'answers' in locals():
        f.write(str(answers))
    elif 'conversation' in locals():
        f.write(str(conversation))
    elif 'generated_text' in locals():
        f.write(str(generated_text))
    elif 'transcription' in locals():
        f.write(str(transcription))
    elif 'text' in locals():
        f.write(str(text))
    else:
        f.write("Task completed.")
print("Saved results.txt")

# 2. Images
if 'image' in locals():
    if hasattr(image, 'save'):
        image.save('generated_image.png')
        print("Saved generated_image.png")
elif 'images' in locals() and isinstance(images, list) and len(images) > 0:
    if hasattr(images[0], 'save'):
        images[0].save('generated_image.png')
        print("Saved generated_image.png (first image)")

# 3. Audio
if 'audio' in locals():
    if isinstance(audio, tuple):
        rate, data = audio
        try:
            wavfile.write('generated_audio.wav', rate, data)
            print("Saved generated_audio.wav")
        except:
            pass
    elif hasattr(audio, 'ndim'): # numpy array
        rate = save_rate if 'save_rate' in locals() else 16000
        try:
            wavfile.write('generated_audio.wav', rate, audio)
            print("Saved generated_audio.wav")
        except:
            pass
elif 'audio_data' in locals():
     rate = save_rate if 'save_rate' in locals() else 16000
     try:
         wavfile.write('generated_audio.wav', rate, audio_data)
         print("Saved generated_audio.wav")
     except:
         pass


## 8. Advanced Usage


**Teaching Tip:** Try varying the input parameters or prompts to see how the model's output changes. This helps build intuition about the model's capabilities and limitations.

In [ ]:
# Try with more confusing labels or different videos from the Kinetics-400 dataset.
print("Try changing the text labels to see how robust the model is!")